# 09 — Retrieval-Augmented Generation (RAG)

This is where everything comes together. **RAG** lets a model answer questions using *your*
documents instead of (or in addition to) what it memorized during training. It's the most
common pattern in production LLM apps: chatbots over company docs, support assistants,
research tools.

The idea is simple:

1. **Retrieve** the chunks most relevant to the user's question (notebook 08).
2. **Augment** the prompt by inserting those chunks as context.
3. **Generate** an answer grounded in that context.

Why bother? Because it solves the two biggest weaknesses of a bare LLM: it doesn't know
your private/recent data, and it can *hallucinate*. Grounding the answer in retrieved text
makes it accurate and lets you **cite sources**.

In this notebook:

1. Rebuild the retriever from our knowledge base.
2. Assemble a RAG chain with LCEL.
3. Prove the answer is grounded (and make it admit when it doesn't know).
4. Return **sources** alongside the answer.
5. Handle **follow-up questions** (conversational RAG).

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."
assert os.path.exists("data/lumina_handbook.md"), "Run `python make_data.py` first."

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
print("Model and embeddings ready.")

## 1. Rebuild the retriever (ingestion recap)

This is the load → split → embed → store pipeline from notebooks 07–08, condensed into one
cell. The result is a `retriever` we can query with a question string.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

docs = DirectoryLoader(
    "data", glob="**/*.md", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_documents(docs)

store = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks. Retriever ready (k=3).")

## 2. The RAG prompt

The prompt is what makes RAG work. It instructs the model to answer **using the provided
context**, and — crucially — to say it doesn't know if the context is insufficient. That
single instruction is your main defense against hallucination.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a support assistant for Lumina Robotics. "
     "Answer the question using ONLY the context below. "
     "If the answer is not in the context, say you don't know. "
     "Be concise.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])
print("Prompt ready.")

## 3. Assemble the RAG chain

The retriever returns a list of `Document`s, but the prompt wants a single string in
`{context}`. So we add a small `format_docs` step to join them. Then LCEL wires it all:

- `{context}` is produced by `retriever | format_docs`,
- `{question}` is the original input, forwarded by `RunnablePassthrough()`.

The chain's input is just the **question string**.

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)

print(rag_chain.invoke("How long is the warranty, and what does it cover?"))

That answer ("24-month limited warranty covering manufacturing defects") came from the
handbook, not the model's training data — the company is fictional. Try a few more.

In [ ]:
for q in [
    "What's the price difference between Pixel One and Pixel Pro?",
    "How do I reset the robot?",
    "How long does the battery last?",
]:
    print("Q:", q)
    print("A:", rag_chain.invoke(q), "\n")

## 4. Grounding: making it say "I don't know"

A grounded assistant should refuse to invent answers. Because our prompt says to use only
the context, a question whose answer isn't in the documents should get a clear "I don't
know" — not a confident fabrication.

In [ ]:
# Nothing in the knowledge base mentions Mars or rockets.
print(rag_chain.invoke("Can Pixel fly to Mars?"))

Compare that to asking the bare model the same thing — without grounding it might
ramble or guess. The retrieved-context-plus-instruction combination is what keeps RAG
honest. (If your model still hallucinates, tighten the system prompt and check that
retrieval is actually returning relevant chunks.)

## 5. Returning sources

Users trust answers more when you show where they came from. Here we restructure the chain
so it returns **both** the answer and the source documents. We run the retriever once,
keep its output, and feed a formatted copy into the answer sub-chain.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Sub-chain that produces just the answer text from {context: [docs], question}.
answer_subchain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | rag_prompt
    | model
    | StrOutputParser()
)

# Retrieve once, keep the docs, and attach the answer.
rag_with_sources = RunnableParallel(
    context=retriever,
    question=RunnablePassthrough(),
).assign(answer=answer_subchain)

out = rag_with_sources.invoke("What is Lumina Cloud and how much does it cost?")
print("ANSWER:", out["answer"], "\n")
print("SOURCES:")
for d in out["context"]:
    print("  -", os.path.basename(d.metadata["source"]))

Now the application layer can display the answer *and* link to the source files —
essential for trust in any real assistant.

## 6. Conversational RAG: handling follow-ups

Real users ask follow-up questions: *"How long is the warranty?"* then *"And what about
returns?"* The second question is meaningless to the retriever on its own — "what about
returns?" has no subject. The fix is a two-step pattern:

1. **Reformulate** the follow-up into a standalone question using the chat history.
2. **Retrieve and answer** with the standalone question.

Let's build the reformulation step first.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the chat history and the latest user question, rewrite the question so it is "
     "fully self-contained and understandable without the history. Do NOT answer it. "
     "If it is already standalone, return it unchanged."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
contextualize = contextualize_prompt | model | StrOutputParser()

history = [
    HumanMessage(content="How long is the Pixel warranty?"),
    AIMessage(content="It's a 24-month limited warranty."),
]
print("Follow-up: 'And what about returns?'")
print("Rewritten:", contextualize.invoke({"history": history, "question": "And what about returns?"}))

The vague follow-up became something like *"What is the return policy for the Pixel?"*
— which the retriever can actually match. Now combine the two steps into one conversational
function: reformulate, retrieve, then answer (with the history available for tone).

In [ ]:
conv_answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Lumina Robotics support assistant. Answer using ONLY the context. "
     "If it's not in the context, say you don't know.\n\nContext:\n{context}"),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

def conversational_rag(question: str, history: list) -> str:
    # 1. Make the question standalone (only needed if there's history).
    standalone = contextualize.invoke({"history": history, "question": question}) if history else question
    # 2. Retrieve with the standalone question.
    docs = retriever.invoke(standalone)
    # 3. Answer, giving the model the context and the history.
    chain = conv_answer_prompt | model | StrOutputParser()
    return chain.invoke({"context": format_docs(docs), "history": history, "question": question})

# Simulate a short conversation:
chat = []
q1 = "How long is the Pixel warranty?"
a1 = conversational_rag(q1, chat)
print("Q1:", q1, "\nA1:", a1, "\n")
chat += [HumanMessage(content=q1), AIMessage(content=a1)]

q2 = "And what about returns?"   # depends on the previous turn
a2 = conversational_rag(q2, chat)
print("Q2:", q2, "\nA2:", a2)

The second answer works because we reformulated "And what about returns?" into a
standalone query before retrieving. This reformulate-then-retrieve pattern is the backbone
of every conversational RAG system.

> **The biggest lever in RAG is retrieval quality.** If answers are wrong, the model is
> usually fine — the *retrieved chunks* were wrong or missing. Before blaming the prompt,
> inspect what `retriever.invoke(question)` actually returned. Tune `chunk_size`, `k`, and
> try `search_type="mmr"`; clean, well-structured source documents help most of all.

## Your turn

Five exercises that exercise a complete RAG system — answering with citations, testing the
"I don't know" guardrail, tuning `k`, a multi-turn conversation, and standing up RAG over your
own facts. Try each in a blank cell before opening its solution.

**Exercise 1 — Answer with citations.** Use `rag_with_sources` to answer "What does the
warranty cover?" and print the answer together with the distinct source files it came from —
the trust-building combo every support bot needs.

*Sample run (illustrative):*

```text
ANSWER: The Pixel comes with a 24-month limited warranty covering manufacturing defects.
CITED : ['lumina_handbook.md']
```

<details><summary>Show solution</summary>

```python
out = rag_with_sources.invoke("What does the warranty cover?")
print("ANSWER:", out["answer"])
print("CITED :", sorted({os.path.basename(d.metadata["source"]) for d in out["context"]}))
```

Showing the answer *and* its sources is what makes users trust a generated response.

</details>

**Exercise 2 — Guardrail test.** Send `rag_chain` one question whose answer is in the docs and
one that clearly isn't (e.g. today's weather). Confirm it answers the first and declines the
second instead of inventing something.

*Sample run (illustrative):*

```text
In-KB    : To reset Pixel, hold the power button for 10 seconds until the light blinks.
Off-topic: I'm sorry, I don't have that information in the provided context.
```

<details><summary>Show solution</summary>

```python
print("In-KB    :", rag_chain.invoke("How do I reset the robot?"))
print("Off-topic:", rag_chain.invoke("What's the weather in Paris today?"))
```

The "use only the context, else say you don't know" instruction is what stops the second
question from producing a confident fabrication.

</details>

**Exercise 3 — Tune `k` for multi-chunk answers.** Ask a question whose answer spans two
chunks (comparing Pixel One vs Pixel Pro battery) with `k=1` and again with `k=4`. See how a
too-small `k` leaves the answer incomplete.

*Sample run (illustrative):*

```text
--- k=1 ---
The Pixel Pro lasts about 10 hours on a charge. I don't have the Pixel One's battery life in
the context.

--- k=4 ---
The Pixel One lasts about 6 hours per charge, while the Pixel Pro lasts about 10 hours.
```

<details><summary>Show solution</summary>

```python
q = "Compare the battery life of Pixel One and Pixel Pro."
for k in (1, 4):
    r = store.as_retriever(search_kwargs={"k": k})
    chain = ({"context": r | format_docs, "question": RunnablePassthrough()}
             | rag_prompt | model | StrOutputParser())
    print(f"--- k={k} ---")
    print(chain.invoke(q), "\n")
```

With `k=1` the model often sees only one product's spec; raising `k` gives it both chunks.

</details>

**Exercise 4 — Multi-turn conversation.** Use `conversational_rag` for a three-turn support
chat where each question leans on the last ("how long is the warranty?" → "and returns?" →
"does that cover accidental damage?").

*Sample run (illustrative):*

```text
Q: How long is the Pixel warranty?
A: It's a 24-month limited warranty.

Q: And what about returns?
A: Returns are accepted within 30 days of delivery for a full refund.

Q: Does that cover accidental damage?
A: No — the warranty covers manufacturing defects, not accidental damage.
```

<details><summary>Show solution</summary>

```python
chat = []
for q in ["How long is the Pixel warranty?",
          "And what about returns?",
          "Does that cover accidental damage?"]:
    a = conversational_rag(q, chat)
    print("Q:", q, "\nA:", a, "\n")
    chat += [HumanMessage(content=q), AIMessage(content=a)]
```

Each follow-up is rewritten into a standalone query before retrieval, so the vague ones still
find the right chunks.

</details>

**Exercise 5 — RAG over your own facts.** Stand up a fresh knowledge base from a few policy
sentences, build the full `retriever | format_docs | prompt | model` chain, and answer a
question — the whole RAG pattern on new data in a few lines.

*Sample run (illustrative):*

```text
Refunds are issued within 5 business days of approval.
```

<details><summary>Show solution</summary>

```python
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate

policy = [
    "Refunds are issued within 5 business days of approval.",
    "Support hours are 9am to 6pm, Monday to Friday.",
    "Free shipping applies to orders over $50.",
]
my_ret = InMemoryVectorStore.from_texts(policy, embedding=embeddings).as_retriever(search_kwargs={"k": 2})

my_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context. If it's not there, say you don't know.\n\nContext:\n{context}"),
    ("human", "{question}"),
])
my_rag = ({"context": my_ret | format_docs, "question": RunnablePassthrough()}
          | my_prompt | model | StrOutputParser())

print(my_rag.invoke("When can I expect my refund?"))
```

Point the same pattern at any documents and you have a grounded assistant for that data.

</details>

## Summary

- **RAG = retrieve relevant chunks, inject them as context, then generate** a grounded answer.
- The **system prompt** must tell the model to use only the context and to admit when it
  doesn't know — this is your main hallucination guard.
- LCEL wires it cleanly:
  `{"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | model | parser`.
- Return **sources** by retrieving once and attaching the answer with `RunnableParallel` +
  `.assign`.
- **Conversational RAG** reformulates follow-ups into standalone questions before retrieving.
- Answer quality is dominated by **retrieval quality** — debug the retriever first.

**Next:** [10 — Tools & Agents](10_Tools_and_Agents.ipynb). Instead of only retrieving text,
we'll let the model *act*: call functions, use a calculator, search — and decide for itself
which to use.